# Auditing LittleLearner's symbolic filter

**Paper:** *LittleLearner: Language Models Under Pedagogically Controlled Knowledge Exposure* (arXiv 2608.13545), Sec. 3.1.4 and App. A.3 / Table 3.

**Claim under test.** The rule-based symbolic filter drops a whole document on *any* regex match and is described as "high-precision" because "recognizable algebraic expressions are typically Beyond-K–5". This notebook measures how well that holds on labelled K-5 and Beyond-K-5 texts:

| | filter **drops** | filter **keeps** |
|---|---|---|
| **Beyond-K-5** (should go) | TP: correctly removed | FN: **leaked** advanced content |
| **K-5** (should stay) | FP: **wrongly removed** elementary content | TN: correctly kept |

**How to use this template.** Cells marked `TODO` are yours: the data, the decisions, and every conclusion. The filter, metrics, and plotting helpers are provided. Set `USE_DEMO_DATA = False` once your own data is in place; the demo rows only exist so the pipeline runs end to end and are **not** results.

Colour code follows the deck: teal = K-5, orange = Beyond-K-5.

## 0. Setup

In [ ]:
import re
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TEAL, ORANGE = "#00939C", "#D9622B"        # K-5, Beyond-K-5 (deck colours, CVD-checked)
INK, MUTED, GRID = "#33414D", "#5B6B78", "#D9E1E6"
LABEL_COLORS = {"K5": TEAL, "BeyondK5": ORANGE}

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "font.size": 10,
})

FIG_DIR = Path("figs")      # figures for slides 17-19 land here
FIG_DIR.mkdir(exist_ok=True)

## 1. The filter, reconstructed from Table 3

The paper does **not** publish its regexes. Table 3 only lists six notation families with representative matches and says each family "also includes equivalent unicode, LaTeX, and textual variants". The patterns below are a reconstruction from that description, extending `code/symbolic_filter_check.py`.

> **TODO (important for the Q&A):** read every pattern, decide whether it is a faithful reading of Table 3, and note deviations in the cell after the unit tests. Your audit measures *this* filter, so the supervisor will ask how close it is to the paper's.

In [ ]:
# Each family is a list of (variant_name, regex). A document is dropped if ANY regex matches.
FILTER_PATTERNS = {
    "Equations": [
        ("single_var",  r"\b\d*[a-z]\s*[+\-]\s*\d+\s*=\s*\d+"),           # 2x + 3 = 7
        ("multi_var",   r"\b\d*[a-z]{1,2}\s*[+\-]\s*\d*[a-z]{1,2}\s*=\s*\w"),  # ax + by = c
        ("polynomial",  r"\b\d*([a-z])\s*[+\-]\s*\d*\1(?:\s*\^|[²³])?"),  # x + 2x, x + 2x^2
    ],
    "Exponents": [
        ("caret",       r"\b\w+\s*\^\s*\{?\w"),                           # x^2, x^{n}, 10^3
        ("unicode",     r"[a-z0-9\)][²³⁴⁵⁶⁷⁸⁹ⁿ]"),                       # x², 5³
        ("digit_base",  r"\b[a-z]\d\b"),                                  # x2 (flattened superscript)
        ("programming", r"\*\*\s*\d|\bpow\("),                            # x**2, pow(x, 2)
    ],
    "Roots": [
        ("symbol",      r"[√∛∜]"),
        ("latex",       r"\\sqrt"),
        ("function",    r"\bsqrt\(|\bcbrt\("),
        ("fractional",  r"\^\s*\(?\s*1\s*/\s*\d"),                        # x^(1/3)
        ("word",        r"\b(?:square|cube|nth|n-th)\s+roots?\b"),
    ],
    "Inequalities": [
        ("chained_lt",  r"\w\s*(?:<|≤|\\le(?:q)?)\s*\w+\s*(?:<|≤|\\le(?:q)?)\s*\w"),
        ("chained_gt",  r"\w\s*(?:>|≥|\\ge(?:q)?)\s*\w+\s*(?:>|≥|\\ge(?:q)?)\s*\w"),
    ],
    "Calculus": [
        ("symbol",      r"[∑Σ∫∂∞∏∇]"),
        ("latex",       r"\\(?:sum|int|partial|infty|prod|lim|nabla)\b"),
        # TODO: \frac is deliberately NOT included (K-5 fractions use it). Your call.
    ],
    "Functions": [
        ("fgh_xyz",     r"\b[fgh]\s*\(\s*[xyz]\s*\)"),                    # restricted per Table 3
    ],
}

FLAGS = 0   # case-sensitive; TODO: decide whether the paper would use re.IGNORECASE
COMPILED = {
    fam: [(name, re.compile(rx, FLAGS)) for name, rx in variants]
    for fam, variants in FILTER_PATTERNS.items()
}

def symbolic_filter(text, patterns=COMPILED):
    # Returns (dropped: bool, hits: list of 'Family/variant' strings, matched snippets).
    hits, snippets = [], []
    for fam, variants in patterns.items():
        for name, rx in variants:
            m = rx.search(text)
            if m:
                hits.append(f"{fam}/{name}")
                snippets.append(m.group(0))
    return bool(hits), hits, snippets

### 1a. Unit tests against Table 3's own examples

In [ ]:
# Every example from Table 3 must be caught; otherwise the reconstruction is too narrow.
TABLE3_EXAMPLES = {
    "Equations":    ["2x + 3 = 7", "ax + by = c", "x + 2x^2"],
    "Exponents":    ["x^2", "x^n", "5^3", "x²"],
    "Roots":        ["√x", "the cube root of 27", "x^(1/3)"],
    "Inequalities": ["a < x < b", "a ≤ x ≤ b"],
    "Calculus":     ["Σ", "∫", "∂", "∞"],
    "Functions":    ["f(x)", "g(y)", "h(z)"],
}
for fam, examples in TABLE3_EXAMPLES.items():
    for ex in examples:
        dropped, hits, _ = symbolic_filter(ex)
        assert any(h.startswith(fam) for h in hits), f"{fam} missed: {ex!r} (hits={hits})"
print("All Table 3 examples are caught.")

# TODO: add a few texts that must NOT be caught (plain prose, '3 + 4 = 7', '1/2 of a pizza', ...)
# and assert they pass. This is where you discover over-broad patterns before the audit.
MUST_KEEP = [
    "The cat sat on the mat.",
]
for t in MUST_KEEP:
    assert not symbolic_filter(t)[0], f"wrongly dropped: {t!r} -> {symbolic_filter(t)[1]}"

**TODO: deviations from the paper.** List each place where you had to guess (e.g. case sensitivity, whether `x2` really means flattened `x²`, whether `\frac` counts as "advanced LaTeX").

- …

## 2. Data

**Unit = whole document / passage**, because the paper drops whole documents. Expected columns:

| column | meaning |
|---|---|
| `id` | unique id |
| `text` | the passage |
| `label` | `K5` or `BeyondK5` (ground truth from the source curriculum) |
| `grade` | integer grade (0 = K), optional but needed for the per-grade plot |
| `subject` | e.g. `math`, `science`, `literature` |
| `source` | where you got it (for the reproducibility slide) |

**TODO: collect the data.** Decide and document:
1. Sources for each label (e.g. CCSS standard texts and openly licensed K-5 vs. middle/high-school materials). Record licence and URL in `source`.
2. How many passages per label and per grade (aim for balanced; note that small n means wide confidence intervals, which section 4 reports).
3. Whether you include non-math text. The symbolic filter is the *last* stage after the classifiers, so its job is notation-heavy math. Measuring leakage on Beyond-K-5 literature would mostly measure what it was never meant to catch. Use `subject` to report both.
4. How you avoid choosing examples that you already know will (not) trigger a pattern.

In [ ]:
USE_DEMO_DATA = True                     # TODO: set to False when your data is ready
DATA_PATH = Path("data/audit_corpus.csv")  # TODO: or .jsonl

REQUIRED = ["id", "text", "label"]

def load_corpus(path):
    path = Path(path)
    if path.suffix == ".jsonl":
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_csv(path)
    missing = [c for c in REQUIRED if c not in df.columns]
    assert not missing, f"missing columns: {missing}"
    assert set(df["label"]) <= {"K5", "BeyondK5"}, f"unexpected labels: {set(df['label'])}"
    for c, default in [("grade", np.nan), ("subject", "unknown"), ("source", "unknown")]:
        if c not in df.columns:
            df[c] = default
    return df.reset_index(drop=True)

def demo_corpus():
    # PLACEHOLDER rows (from the deck's probe script) so the notebook runs. NOT a dataset.
    rows = [
        ("demo1", "Gina has some stickers. She gets 6 more and now has 14. Write an equation: s + 6 = 14.", "K5", 3, "math"),
        ("demo2", "Use whole-number exponents to denote powers of 10: 10^3 = 1,000.", "K5", 5, "math"),
        ("demo3", "The cat sat on the mat. Four cats and two cats make six cats.", "K5", 0, "literature"),
        ("demo4", "Solve 2x + 3 = 7 and simplify x^2 when x = 2.", "BeyondK5", 8, "math"),
        ("demo5", "Photosynthesis converts light energy into chemical energy stored in glucose.", "BeyondK5", 7, "science"),
    ]
    df = pd.DataFrame(rows, columns=["id", "text", "label", "grade", "subject"])
    df["source"] = "DEMO"
    return df

corpus = demo_corpus() if USE_DEMO_DATA else load_corpus(DATA_PATH)
print(f"{len(corpus)} passages", "(DEMO DATA - not results)" if USE_DEMO_DATA else "")
corpus.groupby(["label", "subject"]).size().unstack(fill_value=0)

### 2a. (Optional) sentence- vs document-level

Longer documents are more likely to contain *one* matching string, so the drop rate grows with length. If your passages vary a lot in length, run the audit on fixed-size chunks too and compare.

In [ ]:
def chunk(text, n_sentences=5):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [" ".join(sents[i:i + n_sentences]) for i in range(0, len(sents), n_sentences)]

# TODO (optional): build a chunked corpus and run sections 3-5 on it as well.
# chunked = corpus.assign(text=corpus.text.map(chunk)).explode("text", ignore_index=True)

## 3. Apply the filter

In [ ]:
def apply_filter(df, patterns=COMPILED):
    out = df.copy()
    res = out["text"].map(lambda t: symbolic_filter(t, patterns))
    out["dropped"] = res.map(lambda r: r[0])
    out["hits"] = res.map(lambda r: r[1])
    out["snippets"] = res.map(lambda r: r[2])
    out["n_chars"] = out["text"].str.len()
    return out

audited = apply_filter(corpus)
audited[["id", "label", "grade", "dropped", "hits", "snippets"]]

## 4. Metrics

Positive class = `BeyondK5` (should be dropped); prediction = `dropped`.

- **Wrongly-removed rate** (false positive rate) = FP / (FP + TN): share of K-5 documents lost.
- **Leak rate** (false negative rate) = FN / (TP + FN): share of Beyond-K-5 documents let through.
- **Precision** = TP / (TP + FP): the paper's stated design goal.

With a few dozen passages per class, report 95% Wilson intervals, not bare percentages.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))

def confusion(df):
    y = df["label"].eq("BeyondK5")
    d = df["dropped"]
    return {"TP": int((y & d).sum()), "FN": int((y & ~d).sum()),
            "FP": int((~y & d).sum()), "TN": int((~y & ~d).sum())}

def audit_metrics(df):
    c = confusion(df)
    rows = []
    for name, k, n in [
        ("wrongly_removed_rate (FPR)", c["FP"], c["FP"] + c["TN"]),
        ("leak_rate (FNR)",            c["FN"], c["TP"] + c["FN"]),
        ("recall (TPR)",               c["TP"], c["TP"] + c["FN"]),
        ("precision",                  c["TP"], c["TP"] + c["FP"]),
    ]:
        lo, hi = wilson_ci(k, n)
        rows.append({"metric": name, "k": k, "n": n,
                     "value": k / n if n else np.nan, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows).set_index("metric"), c

metrics, cm = audit_metrics(audited)
print(cm)
metrics.round(3)

In [ ]:
# Per-subject breakdown (see data decision 3 above)
pd.concat({s: audit_metrics(g)[0]["value"] for s, g in audited.groupby("subject")}, axis=1).round(3)

### 4a. Precision at deployment prevalence

Precision on a balanced audit set is not precision on the web corpus, where Beyond-K-5 math notation is rare. Given your measured TPR and FPR, this shows what precision would be at a given prevalence. It is a useful reviewer-style point for the discussion slide.

**TODO:** pick a plausible prevalence range and justify it (the paper reports ~0.1% removal after ModernBERT).

In [ ]:
def precision_at_prevalence(tpr, fpr, prevalence):
    prevalence = np.asarray(prevalence, dtype=float)
    return tpr * prevalence / (tpr * prevalence + fpr * (1 - prevalence))

tpr = metrics.loc["recall (TPR)", "value"]
fpr = metrics.loc["wrongly_removed_rate (FPR)", "value"]
prev = np.logspace(-4, -0.3, 100)       # TODO: adjust range

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(prev, precision_at_prevalence(tpr, fpr, prev), color=ORANGE, lw=2)
ax.set_xscale("log")
ax.set_ylim(0, 1)
ax.set_xlabel("Prevalence of Beyond-K-5 documents")
ax.set_ylabel("Precision")
ax.set_title("Filter precision vs. prevalence", loc="left")
fig.savefig(FIG_DIR / "precision_vs_prevalence.png")
plt.show()

## 5. Plots

In [ ]:
def plot_confusion(c, title="Symbolic filter on audit set"):
    mat = np.array([[c["TP"], c["FN"]], [c["FP"], c["TN"]]])
    fig, ax = plt.subplots(figsize=(4, 3.4))
    ax.imshow(mat, cmap="Greys", vmin=0, vmax=max(mat.max(), 1) * 1.6)
    ax.grid(False)
    ax.set_xticks([0, 1], ["dropped", "kept"])
    ax.set_yticks([0, 1], ["Beyond-K-5", "K-5"])
    for lbl, col in zip(ax.get_yticklabels(), [ORANGE, TEAL]):
        lbl.set_color(col); lbl.set_fontweight("bold")
    names = [["correct", "leaked"], ["wrongly removed", "correct"]]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{mat[i, j]}\n{names[i][j]}", ha="center", va="center", color=INK)
    ax.set_xlabel("Filter decision")
    ax.set_title(title, loc="left")
    return fig

fig = plot_confusion(cm)
fig.savefig(FIG_DIR / "confusion_matrix.png")
plt.show()

In [ ]:
def plot_family_hits(df):
    # Which pattern family fires, split by true label. A family firing on K-5 = source of wrong removals.
    fams = list(FILTER_PATTERNS)
    counts = {lab: Counter() for lab in ["K5", "BeyondK5"]}
    for lab, hits in zip(df["label"], df["hits"]):
        for fam in {h.split("/")[0] for h in hits}:
            counts[lab][fam] += 1
    n = df["label"].value_counts()
    y = np.arange(len(fams))
    h = 0.38
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for k, (lab, name) in enumerate([("K5", "K-5"), ("BeyondK5", "Beyond-K-5")]):
        share = [counts[lab][f] / n.get(lab, 1) for f in fams]
        ax.barh(y + (k - 0.5) * h, share, height=h * 0.9, color=LABEL_COLORS[lab], label=name)
    ax.set_yticks(y, fams)
    ax.invert_yaxis()
    ax.set_xlabel("Share of documents in class where family matches")
    ax.legend(frameon=False, loc="lower right")
    ax.set_title("Which notation families fire", loc="left")
    return fig

fig = plot_family_hits(audited)
fig.savefig(FIG_DIR / "family_hits.png")
plt.show()

In [ ]:
def plot_drop_by_grade(df):
    g = df.dropna(subset=["grade"]).groupby("grade")["dropped"].agg(["sum", "count"])
    if g.empty:
        print("No grade column filled in - skipping."); return None
    rate = g["sum"] / g["count"]
    ci = np.array([wilson_ci(k, n) for k, n in zip(g["sum"], g["count"])])
    colors = [TEAL if gr <= 5 else ORANGE for gr in g.index]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.bar(g.index, rate, color=colors, width=0.7,
           yerr=[rate - ci[:, 0], ci[:, 1] - rate], ecolor=MUTED, capsize=3)
    ax.axvline(5.5, color=MUTED, ls="--", lw=1)
    ax.text(5.4, 1.0, "K-5", ha="right", va="top", color=TEAL, fontweight="bold")
    ax.text(5.6, 1.0, "Beyond", ha="left", va="top", color=ORANGE, fontweight="bold")
    ax.set_xticks(g.index, ["K" if x == 0 else str(int(x)) for x in g.index])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Grade"); ax.set_ylabel("Drop rate (95% CI)")
    ax.set_title("Share of documents dropped, by grade", loc="left")
    return fig

fig = plot_drop_by_grade(audited)
if fig: fig.savefig(FIG_DIR / "drop_by_grade.png"); plt.show()

## 6. Error analysis

In [ ]:
pd.set_option("display.max_colwidth", 200)
wrongly_removed = audited[(audited.label == "K5") & audited.dropped]
leaked = audited[(audited.label == "BeyondK5") & ~audited.dropped]

print(f"Wrongly removed K-5 documents: {len(wrongly_removed)}")
display(wrongly_removed[["id", "grade", "hits", "snippets", "text"]])
print(f"Leaked Beyond-K-5 documents: {len(leaked)}")
display(leaked[["id", "grade", "subject", "text"]])

**TODO: read every error and group them.** For example: K-5 "unknown number" equations (3.OA), powers of ten (5.NBT), chained comparisons (2.NBT), or false matches from ordinary text. For leaks: Beyond-K-5 content written in words rather than symbols, notation outside the six families, etc. One or two concrete examples make the best slide.

- …

## 7. Ablation: drop one family at a time

Shows which family buys recall and which costs elementary content. This answers "is the any-match rule worth it?".

In [ ]:
def ablation(df):
    rows = []
    for fam in ["(none removed)"] + list(COMPILED):
        pats = {k: v for k, v in COMPILED.items() if k != fam}
        m, _ = audit_metrics(apply_filter(df, pats))
        rows.append({"without": fam,
                     "wrongly_removed_rate": m.loc["wrongly_removed_rate (FPR)", "value"],
                     "leak_rate": m.loc["leak_rate (FNR)", "value"]})
    return pd.DataFrame(rows).set_index("without")

abl = ablation(corpus)
abl.round(3)

In [ ]:
def plot_ablation(abl):
    fig, ax = plt.subplots(figsize=(5, 3.6))
    ax.scatter(abl["wrongly_removed_rate"], abl["leak_rate"], s=40, color=INK, zorder=3)
    # variants landing on the same point share one label
    for (x, y), grp in abl.reset_index().groupby(["wrongly_removed_rate", "leak_rate"]):
        ax.annotate("\n".join(grp["without"]), (x, y), xytext=(5, 3),
                    textcoords="offset points", fontsize=8, color=MUTED, va="bottom")
    ax.set_xlabel("Wrongly removed K-5 (FPR)")
    ax.set_ylabel("Leaked Beyond-K-5 (FNR)")
    ax.set_title("Trade-off when removing one family", loc="left")
    return fig

fig = plot_ablation(abl)
fig.savefig(FIG_DIR / "ablation.png")
plt.show()

## 8. Conclusions (TODO)

Write these yourself; they go on slide 19 and into the verdict.

1. **How often does the filter remove elementary content, and why?**
   …
2. **How much advanced content leaks, and is that the filter's fault or by design (it runs after the classifiers)?**
   …
3. **Does the "high-precision" claim hold? Under which prevalence?**
   …
4. **Limitations of this audit** (reconstructed regexes, sample size, source choice, label noise).
   …
5. **One-sentence verdict for the talk.**
   …

---
## Bonus (optional): re-plot pass@k curves from the paper's figures

Idea 3 from the planning. The paper's App. C.2.3 / Fig. 16 shows pass@k on MathCAMPS for LittleLearner vs. Unfiltered. Raw samples are not released, so this only re-plots values **you read off the figure** (e.g. with WebPlotDigitizer). Say so on the slide.

In [ ]:
# TODO: fill with values digitised from the paper. Structure: {model: {grade: {k: pass_at_k_percent}}}
PASSK = {
    "LittleLearner": {},   # e.g. {3: {1: ..., 4: ..., 16: ..., 64: ..., 256: ..., 1024: ...}, 8: {...}}
    "Unfiltered":    {},
}
MODEL_COLORS = {"LittleLearner": TEAL, "Unfiltered": MUTED}

def unbiased_pass_at_k(n, c, k):
    # Chen et al. (2021) estimator, if you ever have n samples with c correct.
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

def plot_passk(passk):
    grades = sorted({g for m in passk.values() for g in m})
    if not grades:
        print("PASSK is empty - fill in digitised values first."); return None
    fig, axes = plt.subplots(1, len(grades), figsize=(3.2 * len(grades), 3), sharey=True, squeeze=False)
    for ax, g in zip(axes[0], grades):
        for model, by_grade in passk.items():
            if g not in by_grade: continue
            ks, vals = zip(*sorted(by_grade[g].items()))
            ax.plot(ks, vals, marker="o", ms=4, lw=2, color=MODEL_COLORS[model], label=model)
            ax.annotate(model, (ks[-1], vals[-1]), xytext=(4, 0), textcoords="offset points",
                        fontsize=8, color=INK, va="center")
        ax.set_xscale("log", base=2)
        ax.set_title(f"Grade {g}", loc="left")
        ax.set_xlabel("k")
    axes[0][0].set_ylabel("pass@k (%)")
    axes[0][0].legend(frameon=False, loc="upper left")
    return fig

fig = plot_passk(PASSK)
if fig: fig.savefig(FIG_DIR / "passk_replot.png"); plt.show()